In [29]:
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

import string
import warnings

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

nltk.download("stopwords")
nltk.download("punkt")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rafatmeraz/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rafatmeraz/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/rafatmeraz/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Data Load

In [30]:
df = pd.read_csv("spam.csv")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [31]:
df.shape

(5572, 5)

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   v1          5572 non-null   str  
 1   v2          5572 non-null   str  
 2   Unnamed: 2  50 non-null     str  
 3   Unnamed: 3  12 non-null     str  
 4   Unnamed: 4  6 non-null      str  
dtypes: str(5)
memory usage: 217.8 KB


## Data Pre-processing

In [33]:
df = df.drop(columns=["Unnamed: 2","Unnamed: 3","Unnamed: 4"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   v1      5572 non-null   str  
 1   v2      5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


In [34]:
df = df.rename(columns={"v1": "target", "v2": "text"})

In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   target  5572 non-null   str  
 1   text    5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


In [36]:
df.duplicated().sum()

np.int64(403)

In [37]:
df = df.drop_duplicates()

In [38]:
df.shape

(5169, 2)

In [39]:
# Label encoder
encoder = LabelEncoder()
df["target"] = encoder.fit_transform(df["target"])

df.sample(2)

,target,text
3815,0,How long does it take to get it.
369,0,"Cool, text me when you're ready"


## Feature Engineering

In [40]:
ps = PorterStemmer()

In [41]:
# transform text into lowercase and procecssing
def transform_text(text):
    text = text.lower()

    text = nltk.word_tokenize(text)

     # Removing special characters
    y = []
    for i in text:
        if i.isalnum():
            y.append(i)

    # Removing stop words and punctuation
    text = y[:]
    y.clear()

    # Loop through the tokens and remove stopwords and punctuation
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)

    # Stemming using Porter Stemmer
    text = y[:]
    y.clear()
    for i in text:
        y.append(ps.stem(i))

    # Join the processed tokens back into a single string
    return " ".join(y)

In [42]:
transform_text("Hello world, I am so happy heute ;)")

'hello world happi heut'

In [43]:
df['transformed_text'] = df['text'].apply(transform_text)

df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [44]:
tfidf = TfidfVectorizer(max_features=300)

X = tfidf.fit_transform(df['transformed_text']).toarray()
y = df['target'].values

In [45]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Model Training

In [46]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier, \
    GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

models = {
    "svc": SVC(kernel="linear", C=1.0, gamma="auto"),
    "lr": LogisticRegression(solver="liblinear", C=1.0, max_iter=1000),
    "dtc": DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=5),
    "rfc": RandomForestClassifier(max_depth=5, min_samples_split=5),
    "abc": AdaBoostClassifier(n_estimators=100, learning_rate=0.1),
    "bc": BaggingClassifier(n_estimators=50, random_state=2),
    "etc": ExtraTreesClassifier(n_estimators=50, random_state=2),
    "xgbc": XGBClassifier(n_estimators=50, random_state=2),
    "gbc": GradientBoostingClassifier(n_estimators=50, random_state=2)
}

In [47]:
from sklearn.metrics import accuracy_score, precision_score

def train_classifier(clf, X_train, y_train, X_test, y_test):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    return accuracy, precision

In [48]:
accuracy_scores = []
precision_scores = []

for name, model in models.items():
    accuracy, precision = train_classifier(model, X_train, y_train, X_test, y_test)
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)

    print(f"For Model: {name}")
    print(f"Accuracy: {accuracy}")
    print(f"Precision: {precision}")

For Model: svc
Accuracy: 0.9700193423597679
Precision: 0.9453125
For Model: lr
Accuracy: 0.965183752417795
Precision: 0.957983193277311
For Model: dtc
Accuracy: 0.9264990328820116
Precision: 0.7948717948717948
For Model: rfc
Accuracy: 0.9032882011605415
Precision: 1.0
For Model: abc
Accuracy: 0.8752417794970987
Precision: 1.0
For Model: bc
Accuracy: 0.9535783365570599
Precision: 0.8702290076335878
For Model: etc
Accuracy: 0.9700193423597679
Precision: 0.9253731343283582
For Model: xgbc
Accuracy: 0.9622823984526112
Precision: 0.9140625
For Model: gbc
Accuracy: 0.9410058027079303
Precision: 0.9375
